In [ ]:
# 1. Imports and dataset path
import json
import os
import random
import re
import spacy
import pandas as pd

DATA_PATH = "dataset_6entities_english.json"

print("Using:", DATA_PATH)
print("File exists:", os.path.exists(DATA_PATH))


Using: dataset_6entities_english.json
File exists: False


In [5]:
# 2. Load the completed 6-entity dataset
with open(DATA_PATH, "r", encoding="utf-8") as f:
    records = json.load(f)

print("Number of records:", len(records))
print("First record keys:", records[0].keys())


Number of records: 300
First record keys: dict_keys(['doc_id', 'text', 'annotations'])


In [6]:
# 3. Check the six entity labels
TARGET_LABELS = [
    "SYMPTOM",
    "MEDICATION",
    "TREATMENT",
    "DIAGNOSIS",
    "CONDITION",
    "EXAM"
]

label_counts = {}
for record in records:
    for ann in record["annotations"]:
        label_counts[ann["label"]] = label_counts.get(ann["label"], 0) + 1

print("Entity counts:")
for label in TARGET_LABELS:
    print(f"{label}: {label_counts.get(label, 0)}")


Entity counts:
SYMPTOM: 1609
MEDICATION: 1300
TREATMENT: 173
DIAGNOSIS: 185
CONDITION: 1185
EXAM: 851


In [7]:
# 4. Convert JSON annotations to spaCy training examples
from spacy.training import Example

tokenizer_nlp = spacy.blank("pt")

def make_examples(records):
    examples = []
    snapped = 0    # spans adjusted to token boundaries
    dropped = 0    # spans that were invalid or unrecoverable

    for record in records:
        text = record["text"]
        ref_doc = tokenizer_nlp.make_doc(text)

        spans = []
        for ann in record["annotations"]:
            label = ann["label"]
            if label not in TARGET_LABELS:
                continue

            start, end = int(ann["begin"]), int(ann["end"])
            if not (0 <= start < end <= len(text)):
                dropped += 1
                continue

            # "expand" snaps partial-token offsets outward to full tokens
            span = ref_doc.char_span(start, end, label=label, alignment_mode="expand")
            if span is None:
                dropped += 1
                continue
            if (span.start_char, span.end_char) != (start, end):
                snapped += 1
            spans.append(span)

        # Remove overlaps AFTER snapping (expansion can create new overlaps); keep longest first
        spans.sort(key=lambda s: (s.start, -(s.end - s.start)))
        accepted, last_end = [], 0
        for s in spans:
            if s.start >= last_end:
                accepted.append(s)
                last_end = s.end

        ref_doc.set_ents(accepted)
        pred_doc = tokenizer_nlp.make_doc(text)
        examples.append(Example(pred_doc, ref_doc))

    return examples, snapped, dropped

examples, snapped, dropped = make_examples(records)

print("Examples created:", len(examples))
print("Spans snapped to token boundaries:", snapped)
print("Spans dropped:", dropped)

Examples created: 300
Spans snapped to token boundaries: 23
Spans dropped: 0


In [8]:
# 5. Train/test split
random.seed(42)
random.shuffle(examples)

split = int(len(examples) * 0.70)

TRAIN_DATA = examples[:split]
TEST_DATA = examples[split:]

print("Training records:", len(TRAIN_DATA))
print("Testing records:", len(TEST_DATA))


Training records: 210
Testing records: 90


In [9]:
# 6. Create the spaCy NER model
nlp = spacy.blank("pt")

ner = nlp.add_pipe("ner")

for label in TARGET_LABELS:
    ner.add_label(label)

optimizer = nlp.begin_training()

print("Labels added:")
print(TARGET_LABELS)


Labels added:
['SYMPTOM', 'MEDICATION', 'TREATMENT', 'DIAGNOSIS', 'CONDITION', 'EXAM']


In [10]:
# 7. Train the model
from spacy.util import minibatch, compounding

N_ITER = 30

for epoch in range(N_ITER):
    random.shuffle(TRAIN_DATA)
    losses = {}

    batches = minibatch(TRAIN_DATA, size=compounding(2.0, 8.0, 1.001))

    for batch in batches:
        nlp.update(
            batch,
            drop=0.25,
            losses=losses
        )
        

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{N_ITER} - Losses: {losses}")


Epoch 5/30 - Losses: {'ner': np.float32(898.21423)}
Epoch 10/30 - Losses: {'ner': np.float32(516.75714)}
Epoch 15/30 - Losses: {'ner': np.float32(401.8557)}
Epoch 20/30 - Losses: {'ner': np.float32(296.18567)}
Epoch 25/30 - Losses: {'ner': np.float32(264.9126)}
Epoch 30/30 - Losses: {'ner': np.float32(202.92859)}


In [13]:
# 8. Evaluate on the test set
scores = nlp.evaluate(TEST_DATA)

print("Precision:", round(scores["ents_p"], 3))
print("Recall:   ", round(scores["ents_r"], 3))
print("F1:       ", round(scores["ents_f"], 3))

print("\nPer-entity scores:")
for label, s in scores["ents_per_type"].items():
    print(f"{label:12} P={s['p']:.3f}  R={s['r']:.3f}  F1={s['f']:.3f}")


Precision: 0.946
Recall:    0.935
F1:        0.94

Per-entity scores:
DIAGNOSIS    P=0.372  R=0.258  F1=0.305
CONDITION    P=0.945  R=0.997  F1=0.970
EXAM         P=0.993  R=0.996  F1=0.994
TREATMENT    P=0.958  R=1.000  F1=0.979
SYMPTOM      P=0.982  R=0.996  F1=0.989
MEDICATION   P=0.929  R=0.858  F1=0.892


In [14]:
# 9. Save the trained model
MODEL_PATH = "modeloNER_6_entidades"

nlp.to_disk(MODEL_PATH)

print("Model saved to:", MODEL_PATH)


Model saved to: modeloNER_6_entidades


In [15]:
# 10. Test the trained NER model on one record
sample_text = records[0]["text"]

doc = nlp(sample_text)

print("Entities found:")
for ent in doc.ents:
    print(f"{ent.label_:12} -> {ent.text}")


Entities found:
SYMPTOM      -> dor abdominal intensa
SYMPTOM      -> vómitos
SYMPTOM      -> dor abdominal
SYMPTOM      -> vómitos
SYMPTOM      -> perda de apetite
SYMPTOM      -> febre
CONDITION    -> doença cardiovascular
CONDITION    -> diabetes tipo 2
TREATMENT    -> regime alimentar
EXAM         -> Análises
EXAM         -> Ecografia abdominal
CONDITION    -> Cirrose hepática
EXAM         -> Rx torax
SYMPTOM      -> edema
EXAM         -> Rx abdominal
CONDITION    -> Diabetes Mellitus tipo 2
TREATMENT    -> insulinotratado
CONDITION    -> Doença cardiovascular
EXAM         -> ECG
EXAM         -> ETT
CONDITION    -> Dislipidemia
CONDITION    -> Hipertensão arterial
MEDICATION   -> Insulina Novomix 60 U ao pequeno-almoço + 40 U ao jantar
MEDICATION   -> Metformina + Dapagliflozina 1000+5 mg 1 cp pequeno-almoço + 1 cp jantar
MEDICATION   -> Carvedilol 6.25mg, 1 cp pequeno-almoço + 1 cp à noite
MEDICATION   -> Espironolactona 25mg, 1cp ao pequeno-almoço
MEDICATION   -> Furosemida 1 cp 

In [16]:
# 11. Extract the six entities for ALL 300 records
output_rows = []

for record in records:
    text = record["text"]
    doc = nlp(text)

    row = {
        "doc_id": record["doc_id"],
        "symptom": [],
        "medication": [],
        "treatment": [],
        "diagnosis": [],
        "condition": [],
        "exam": []
    }

    for ent in doc.ents:
        column = ent.label_.lower()
        if column in row:
            if ent.text not in row[column]:
                row[column].append(ent.text)

    # Convert lists to readable strings
    for column in ["symptom", "medication", "treatment", "diagnosis", "condition", "exam"]:
        row[column] = "; ".join(row[column])

    output_rows.append(row)

result_df = pd.DataFrame(output_rows)

print(result_df.head())


   doc_id                                            symptom  \
0       1  dor abdominal intensa; vómitos; dor abdominal;...   
1       2  Dor abdominal; dor abdominal intensa; edema; h...   
2       3                                                      
3       4        dispneia; dor torácica; febre; tosse; edema   
4       5                  tosse; febre; dor torácica; edema   

                                          medication  \
0  Insulina Novomix 60 U ao pequeno-almoço + 40 U...   
1  Metformina 1000mg bid; Liraglutido 1.8 (Victoz...   
2  Enalapril 20mg id; Amlodipino 10mg id; Simvast...   
3  losartan 50 mg/dia; metformina 850 mg 2x/dia; ...   
4  Salbutamol quando necessário; Salmeterol/fluti...   

                           treatment                          diagnosis  \
0  regime alimentar; insulinotratado                                      
1                     Antibioterapia                                      
2                                                    

In [17]:
# 12. Binary feature matrix: 1 = entity present, 0 = absent
entity_columns = [
    "symptom",
    "medication",
    "treatment",
    "diagnosis",
    "condition",
    "exam"
]

binary_df = result_df[["doc_id"]].copy()

for column in entity_columns:
    binary_df[column] = result_df[column].apply(
        lambda x: 1 if str(x).strip() else 0
    )

print(binary_df.head())


   doc_id  symptom  medication  treatment  diagnosis  condition  exam
0       1        1           1          1          0          1     1
1       2        1           1          1          0          1     1
2       3        0           1          0          0          1     1
3       4        1           1          1          1          1     1
4       5        1           1          0          1          1     1


In [18]:
# 13. Save the NER output
result_df.to_csv(
    "NER_output_6_entities.csv",
    index=False,
    encoding="utf-8-sig"
)

binary_df.to_csv(
    "NER_binary_matrix_6_entities.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print("- NER_output_6_entities.csv")
print("- NER_binary_matrix_6_entities.csv")


Saved:
- NER_output_6_entities.csv
- NER_binary_matrix_6_entities.csv
